# 03. Chronos-2 configurations.

Each one runs on the same 10 windows without overlap, the last 20 days. All of them use the actual
weather of the window, so the comparison between them is fair, while the absolute scores are optimistic.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common import *

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 160, "display.max_columns", 40)
data, weather_fc = load_data()
y = data["y"].to_numpy(float)
FEATURES = [c for c in weather_fc.columns if c != "ds"]
print(len(data), "hours, last one:", data["ds"].iloc[-1])

import time
pipe = load_chronos()
WINDOWS = 10
CORE = ["cloud_cover", "temperature", "wind_speed", "humidity"]     # the columns DynReg used
NO_DERIVED = [c for c in FEATURES if c not in ("air_density", "daylight")]   # both follow from the others

CONFIGS = {
    "base: 8192 h, all weather": dict(features=FEATURES, context=8192),   # 8192 is the model's maximum
    "4096 h, all weather": dict(features=FEATURES, context=4096),
    "2048 h, all weather": dict(features=FEATURES, context=2048),
    "1024 h, all weather": dict(features=FEATURES, context=1024),
    "512 h, all weather": dict(features=FEATURES, context=512),
    "8192 h, no weather": dict(features=[], context=8192),
    "8192 h, core weather": dict(features=CORE, context=8192),
    "8192 h, no derived columns": dict(features=NO_DERIVED, context=8192),
    "8192 h, all weather, log y": dict(features=FEATURES, context=8192, log=True),
}
print(len(CONFIGS), "configurations,", WINDOWS, "windows")

33480 hours, last one: 2026-10-27 00:00:00


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights:  17%|█▋        | 29/170 [00:00<00:00, 278.15it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 955.38it/s]

9 configurations, 10 windows


In [2]:
frames, seconds = [], {}
for name, setting in CONFIGS.items():
    t0 = time.time()
    for k, past_df, test_df in origins(data, WINDOWS):
        out = chronos_predict(pipe, past_df, test_df, **setting)
        frame = pd.DataFrame({"config": name, "window": k, "step": np.arange(1, H + 1), "daylight": test_df["daylight"].to_numpy(),
                              "actual": test_df["y"].to_numpy(float), "point": out["point"], "median": out["median"],
                              "lo": out["lo"], "hi": out["hi"]})
        frames.append(frame)
    seconds[name] = time.time() - t0
    print(f"{name}: {seconds[name]:.0f} s")

results = pd.concat(frames, ignore_index=True)
bench = pd.concat([pd.DataFrame({"window": k, "step": np.arange(1, H + 1), "benchmark": snaive(p["y"])[0],
                                 "benchmark lo": snaive(p["y"])[1], "benchmark hi": snaive(p["y"])[2]})
                   for k, p, t in origins(data, WINDOWS)], ignore_index=True)
results = results.merge(bench, on=["window", "step"])
results.to_csv("chronos_configs.csv", index=False)

base: 8192 h, all weather: 109 s


4096 h, all weather: 41 s


2048 h, all weather: 18 s


1024 h, all weather: 16 s


512 h, all weather: 9 s


8192 h, no weather: 15 s


8192 h, core weather: 73 s


8192 h, no derived columns: 85 s


8192 h, all weather, log y: 111 s


## Day 2 scores

Hours 25-48 decide the ranking, so that is what we compare. The last three columns count windows out of 10
in which the configuration beats the benchmark head to head.

In [3]:
def day2_table(df, point="point"):
    rows = {}
    for name, part in df[df["step"] > 24].groupby("config", sort=False):
        windows = []
        for k, g in part.groupby("window"):
            ours = score(g["actual"], g[point], g["lo"], g["hi"])
            base = score(g["actual"], g["benchmark"], g["benchmark lo"], g["benchmark hi"])
            windows.append({**ours, **{f"wins {m}": int(ours[m] < base[m]) for m in ("RMSE", "MAPE", "Winkler")}})
        means = pd.DataFrame(windows)
        rows[name] = {**means[["RMSE", "MAPE", "Winkler", "coverage"]].mean().round(2).to_dict(),
                      **means[["wins RMSE", "wins MAPE", "wins Winkler"]].sum().astype(int).to_dict()}
    return pd.DataFrame(rows).T

table = day2_table(results)
table

,RMSE,MAPE,Winkler,coverage,wins RMSE,wins MAPE,wins Winkler
"base: 8192 h, all weather",210.89,109.79,1101.59,0.98,7.0,7.0,6.0
"4096 h, all weather",226.42,124.23,1061.34,0.97,7.0,7.0,6.0
"2048 h, all weather",236.35,131.98,1052.00,0.98,7.0,7.0,6.0
"1024 h, all weather",234.30,135.37,1070.26,0.99,7.0,7.0,6.0
"512 h, all weather",226.47,131.16,1164.79,0.99,7.0,7.0,5.0
"8192 h, no weather",285.70,256.52,1446.91,1.00,7.0,5.0,4.0
"8192 h, core weather",221.88,111.56,1063.41,0.98,7.0,7.0,6.0
"8192 h, no derived columns",220.05,110.00,1073.24,0.98,7.0,7.0,6.0
"8192 h, all weather, log y",263.89,95.25,1230.51,0.99,6.0,6.0,5.0


## Is the point forecast a mean or a median?

This decides whether a separate median run is worth testing, so we compare `predictions` with the 0.5 quantile.

In [4]:
base = results[results["config"] == "base: 8192 h, all weather"]
print("largest gap between `predictions` and the 0.5 quantile:", round(float((base["point"] - base["median"]).abs().max()), 6))

largest gap between `predictions` and the 0.5 quantile: 0.0


## Night hours from the log run

The log run should help MAPE on small night values and hurt on the daytime peak. So we try it at night only.

In [5]:
log_run = results[results["config"] == "8192 h, all weather, log y"].set_index(["window", "step"])["point"]
hybrid = base.copy()
hybrid["point"] = np.where(hybrid["daylight"] == 0, log_run.reindex(pd.MultiIndex.from_frame(hybrid[["window", "step"]])).to_numpy(), hybrid["point"])
hybrid["config"] = "base, log run at night"
day2_table(pd.concat([base, hybrid]))

,RMSE,MAPE,Winkler,coverage,wins RMSE,wins MAPE,wins Winkler
"base: 8192 h, all weather",210.89,109.79,1101.59,0.98,7.0,7.0,6.0
"base, log run at night",214.24,91.83,1101.59,0.98,7.0,7.0,6.0


## Differences by window

The mean can hide a single extreme window, so we also ask in how many windows a configuration beats the base case.

In [6]:
day2 = results[results["step"] > 24]
by_window = day2.groupby(["config", "window"], sort=False).apply(
    lambda g: pd.Series({"RMSE": rmse(g["actual"], g["point"]), "MAPE": mape(g["actual"], g["point"]),
                         "Winkler": winkler(g["actual"], g["lo"], g["hi"])}), include_groups=False)
reference = by_window.loc["base: 8192 h, all weather"]
versus = {name: {f"better {m}": int((group.droplevel(0)[m] < reference[m]).sum()) for m in ("RMSE", "MAPE", "Winkler")}
          for name, group in by_window.groupby(level=0, sort=False) if not name.startswith("base")}
versus = pd.DataFrame(versus).T
versus["median RMSE"] = by_window["RMSE"].groupby(level=0, sort=False).median().round(1)
versus["median MAPE"] = by_window["MAPE"].groupby(level=0, sort=False).median().round(1)
versus

,better RMSE,better MAPE,better Winkler,median RMSE,median MAPE
"4096 h, all weather",2,4,9,211.8,91.4
"2048 h, all weather",3,5,7,218.8,78.3
"1024 h, all weather",4,4,5,246.3,77.2
"512 h, all weather",3,6,2,204.6,76.4
"8192 h, no weather",3,2,1,326.1,91.3
"8192 h, core weather",4,3,8,236.5,125.6
"8192 h, no derived columns",4,5,9,229.3,107.7
"8192 h, all weather, log y",1,4,2,260.7,68.5


## What this means

- **Why 8192 hours.** It is the longest context the model accepts. Among the context lengths it has the best RMSE
  (211) and MAPE (110) on these windows. It was first set to 2048 without a reason, and this comparison changed it.
- **What it costs.** On these 10 windows Winkler is worse than at 2048 (1102 against 1052), and 2048 is better in
  7 windows of 10. On 30 windows of day 2 the means favour 8192 on all three metrics: RMSE 196 against 204,
  MAPE 71 against 79, Winkler 978 against 983, though it has the better Winkler in only 11 windows. RMSE and MAPE
  carry 60% of the ranking and the interval is capped anyway, so 8192 stays.
- **Weather is the setting that matters most.** Without it RMSE is 286, MAPE 257, Winkler 1447.
- **Fewer weather columns.** The four core columns (RMSE 222, MAPE 112) and the set without derived columns (220, 110)
  lose on RMSE and have a better Winkler (1063 and 1073), in 8 and 9 windows of 10. All columns stay for the point
  forecast. The gaps are small, and with 10 windows they can be chance.
- **Log target.** MAPE falls to 95, but RMSE rises to 264 and Winkler to 1231, so it is rejected.
- **Log run at night.** It moves MAPE from 110 to 92 and RMSE from 211 to 214 here. An earlier 30-window check at
  2048 hours found the MAPE gain came from a few extreme windows, so the point forecast stays simple.
- **Mean or median.** `predictions` equals the 0.5 quantile, so there is no separate median forecast to try.